# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and analyze the FAIR\(^2\) dataset using the `mlcroissant` library.

### Dataset Source
The dataset Croissant schema is accessible at:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure mlcroissant is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the Croissant dataset
ds = mlc.Dataset(croissant_url)

metadata = ds.metadata
print(f"Name: {metadata.name}\nDescription: {metadata.description}")

## 2. Data Overview
Review available record sets and explore their field and column `@id`s.

In [ ]:
# List all record sets by their @id
print('Available record sets:')
for rs in ds.list_recordsets():
    print(f"- @id: {rs['@id']} | name: {rs.get('name', '(no name)')}")

# Choose a main record set for demonstration (from inspection, the main table is in the following @id)
# e.g., for Croissant v1.0, it's often 'cr:RecordSet/0' or similar. We'll iterate and pick the first main table.
record_sets = [rs['@id'] for rs in ds.list_recordsets()]
assert len(record_sets) > 0, 'No record sets found!'

main_recordset_id = record_sets[0]  # You may adjust if you want another
print(f'\nExamining fields/columns for record set @id: {main_recordset_id}\n')

# List columns for chosen record set
fields = ds.get_recordset_fields(main_recordset_id)
print("Fields for this record set:")
for idx, f in enumerate(fields):
    print(f"  {idx}: name={f.get('name','')}  | @id={f['@id']}  | dataType={f.get('dataType','')}" )

## 3. Data Extraction
Load tabular data from the main record set(s) into pandas DataFrames. We reference record and field entities by their `@id` fields as shown in the overview.

In [ ]:
# Load data for all available record sets into DataFrames
dfs = {}
for record_set_id in record_sets:
    print(f"Loading records for {record_set_id} ...")
    records_iter = ds.records(record_set=record_set_id)
    records = list(records_iter)
    if len(records) > 0:
        dfs[record_set_id] = pd.DataFrame(records)
        print(f"  Loaded shape: {dfs[record_set_id].shape}")
    else:
        print("  (No records found)")

# We'll display the first record set DataFrame's columns as example
print(f"\nColumns in DataFrame for record_set @id {main_recordset_id}:")
print(dfs[main_recordset_id].columns.tolist())

# Show preview
dfs[main_recordset_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and summarizing data. We use field/column `@id`s for referencing, as previously listed.

In [ ]:
#--- Choose a numeric column (by @id) to analyze. The typical columns are 'cr:Field/6' (e.g., 'Abundance_sample1') or similar.
# You can list dfs[main_recordset_id].columns to see all @ids.
print('Columns available:')
print(dfs[main_recordset_id].columns.tolist())

# For demonstration, suppose the main abundance numeric field is '@id' == 'cr:Field/6' (adjust as needed if your field list is different)
numeric_field_id = dfs[main_recordset_id].columns[6] # e.g., 'cr:Field/6' -- change if order is different

print(f"\nAnalyzing field: {numeric_field_id}")

# Remove rows with missing or non-numeric values in the selected column
df_numeric = dfs[main_recordset_id].copy()
df_numeric = df_numeric[pd.to_numeric(df_numeric[numeric_field_id], errors='coerce').notnull()]
df_numeric[numeric_field_id] = df_numeric[numeric_field_id].astype(float)

# Filter: keep records with numeric_field > threshold
threshold = 10
filtered_df = df_numeric[df_numeric[numeric_field_id] > threshold]
print(f"Filtered records where field @{numeric_field_id} > {threshold}:")
print(filtered_df.head())

# Normalize the selected numeric field (z-score scaling)
filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"\nHead of normalized values for {numeric_field_id} (z-score):")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# If categorical fields exist, group by them--pick a likely categorical @id
group_field_id = dfs[main_recordset_id].columns[1]  # e.g., 'cr:Field/1' could be Description or Category
if group_field_id in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
    print(f"\nMean {numeric_field_id} grouped by {group_field_id}:")
    print(grouped_df.head())

## 5. Visualization
Visualize the distribution of the selected numeric field and comparisons across groups (if applicable).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Distribution histogram
plt.figure(figsize=(8,4))
sns.histplot(filtered_df[numeric_field_id], kde=True)
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.ylabel("Count")
plt.show()

# If grouping field exists, plot mean by group
if group_field_id in filtered_df.columns:
    plt.figure(figsize=(10,5))
    order = grouped_df.sort_values(numeric_field_id, ascending=False)[group_field_id]
    sns.barplot(data=grouped_df, x=group_field_id, y=numeric_field_id, order=order)
    plt.xticks(rotation=90)
    plt.title(f"Mean {numeric_field_id} by {group_field_id}")
    plt.show()

## 6. Conclusion
- The dataset provides mass spectrometry-based protein abundance and characteristics for human mast cell extracellular vesicle proteomics.
- Using `mlcroissant`, we explored available record sets, fields, loaded the primary data, applied filters, normalization, grouping, and visualized the distribution of abundance metrics.
- For specific scientific questions, select fields and processing criteria according to your needs using field `@id`s obtained from the Croissant schema.